# Bobby Carrot — PPO + ICM Training (T4 GPU)

Train step-by-step on **normal levels 1–7**, test generalization on **levels 8–10**.

This notebook splits the curriculum into three explicit phases so you can monitor success before proceeding:
- **Phase 1**: Levels 1-3 (basic navigation & carrot collection)
- **Phase 2**: Levels 1-5 (adds crumble tiles)
- **Phase 3**: Levels 1-7 (full training set)

Each phase ends with an **automatic evaluation** so you can verify improvement.

## 1. Setup — Clone & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Charan20510/Mini_Project_Game.git /content/Mini_Project_Game
%cd /content/Mini_Project_Game

In [ ]:
%pip install torch numpy pandas matplotlib

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Game')
os.getcwd()

In [ ]:
!cp -r /content/drive/MyDrive/Game/checkpoints_phase1 /content/Mini_Project_Game/
!cp -r /content/drive/MyDrive/Game/logs_phase1 /content/Mini_Project_Game/

In [ ]:
!ls /content/Mini_Project_Game
%cd /content/Mini_Project_Game

import os
os.getcwd()

## 2. Phase 1 — Pre-trained Checkpoints (Levels 1–3)

**Phase 1 training is skipped** — pre-trained checkpoints from levels 1–3 have been copied from Google Drive in the setup step above.

The cell below verifies the checkpoints are present before proceeding to Phase 2.

In [ ]:
import os

ckpt_best  = 'checkpoints_phase1/ppo/ppo_best.pt'
ckpt_final = 'checkpoints_phase1/ppo/ppo_final.pt'

if os.path.exists(ckpt_best):
    size_mb = os.path.getsize(ckpt_best) / 1e6
    print(f'✅ Found ppo_best.pt  ({size_mb:.1f} MB)')
    phase1_ckpt = ckpt_best
elif os.path.exists(ckpt_final):
    size_mb = os.path.getsize(ckpt_final) / 1e6
    print(f'✅ Found ppo_final.pt ({size_mb:.1f} MB) — best not available, using final')
    phase1_ckpt = ckpt_final
else:
    raise FileNotFoundError(
        'Phase 1 checkpoint not found. '
        'Make sure checkpoints_phase1/ was copied from Drive correctly.'
    )

print(f'Phase 1 checkpoint ready: {phase1_ckpt}')
print('Proceeding directly to Phase 2 training...')

## 3. Phase 2 — Train on Levels 1 to 5 (Anti-Forgetting Retrain)

Retrain Phase 2 from the **Phase 1** checkpoint with anti-forgetting guards.

Previous Phase 2 run collapsed on L2/L3 the moment L4 entered the curriculum (L2: 77% → 0%, L3: 70% → 0%). Root cause: weighted sampling dropped mastered levels to 0.40 while failing L4 pulled weight up to ~1.0. Fixes applied below — none of which leak test-level data or hardcode per-map behaviour.

**Anti-forgetting additions (`TrainingConfig`):**
- `curriculum_mastery_floor=0.60` — mastered levels keep ≥0.60 sampling weight (was 0.40)
- `curriculum_min_quota=0.15` — hard floor: every active level gets ≥15% of rollout episodes
- `level_history_window=60` — wider per-level success window → less flapping (was 30)
- `curriculum_dwell_windows=2` — require 2 consecutive windows above threshold before promotion
- `entropy_boost_steps=50_000` — temporary 2× entropy after promotion, forces exploration on the new level
- `lr_decay_final_fraction=0.3` — cosine LR decay over last 30% so late updates don't re-break L1–L3

**Env-level fixes (applied in `rl_env.py` defaults — no per-level tuning):**
- Stronger revisit / backtrack penalties (counter the "wandering after collection" pattern seen on L1)
- `loop_window` widened 12 → 32 so revisit detection covers larger level loops
- `distance_delta_scale` 0.5 → 0.3 and `finish_approach_bonus` 0.3 → 0.15 to reduce over-reliance on dense gradient pull
- Channel 11 deduped (only `tile==20`; `tile==46` is already on channel 5)

**Resume settings (unchanged from previous Phase 2):**
- `reset_policy_head_on_resume=False` — action semantics identical, keep Phase 1 policy
- `max_steps=400`, `clip_ratio=0.15`, `minibatch=128`, ICM disabled
- **Resume from `checkpoints_phase1/ppo/ppo_best.pt`, NOT from the collapsed Phase 2 checkpoint**

In [ ]:
import os
from pathlib import Path
from Bobby_Carrot.rl_models.config import PPOConfig, TrainingConfig, ICMConfig, LevelConfig
from Bobby_Carrot.rl_models.ppo import train_ppo

level_config_2 = LevelConfig(
    train_levels=[("normal", i) for i in range(1, 6)],   # L1-5 available; curriculum gates which are active
    test_levels=[("normal", i) for i in range(1, 6)],
)
train_config_2 = TrainingConfig(
    total_timesteps=800_000,
    device="auto",
    checkpoint_dir=Path("checkpoints_phase2"),
    log_dir=Path("logs_phase2"),
    curriculum=True,
    curriculum_start_levels=3,                  # Start L1-L3; unlock one level at a time
    curriculum_promotion_window=30,             # Per-level window used by promotion gate
    curriculum_promotion_threshold=0.70,        # 70% success on highest active level to promote
    curriculum_add_levels=1,
    # --- Anti-forgetting guards (Phase 2 fix) ---
    level_history_window=60,                    # Widened from 30 — less variance in per-level signal
    curriculum_mastery_floor=0.60,              # Mastered levels keep ≥0.60 weight (was 0.40)
    curriculum_min_quota=0.15,                  # Hard floor: every active level ≥15% of rollout
    curriculum_dwell_windows=2,                 # Need 2 consecutive windows at threshold to promote
    entropy_boost_steps=50_000,                 # After promotion, 2× entropy for 50k steps
    entropy_boost_multiplier=2.0,
    lr_decay_final_fraction=0.3,                # Cosine LR decay over last 30%
    lr_decay_min_multiplier=0.3,
    # --- Unchanged Phase 2 settings ---
    observation_mode="full",
    max_steps_per_episode=400,                  # Caps wandering; prevents reward-hacking on L4/L5
    reward_scale=1.0,
    reset_policy_head_on_resume=False,          # CRITICAL: keep Phase 1 policy head (action space identical)
)
ppo_config_2 = PPOConfig(
    lr=1e-4,
    entropy_coeff=0.10,
    entropy_min=0.05,
    clip_ratio=0.15,                            # Tighter clip on resume to preserve Phase 1 policy
    rollout_length=2048,
    n_epochs=4,
    minibatch_size=128,
    cnn_channels=[32, 64, 64, 64],
)
# ICM disabled — dense env rewards already handle exploration; curiosity amplifies reward-hacking.
icm_config_2 = ICMConfig(enabled=False)

# Resume from Phase 1 (NOT from the collapsed checkpoints_phase2).
resume_1 = "checkpoints_phase1/ppo/ppo_best.pt"
if not os.path.exists(resume_1):
    resume_1 = "checkpoints_phase1/ppo/ppo_final.pt"

print(f"Loading agent from Phase 1: {resume_1}")
print("Starting Phase 2 Training with anti-forgetting guards (L1→L5, 800k steps)...")
agent_2 = train_ppo(
    ppo_config_2, train_config_2, level_config_2, icm_config_2,
    resume_path=resume_1,
)

### Phase 2 — Evaluation
Check performance on all 5 training levels.

In [ ]:
import os
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

ckpt_2 = 'checkpoints_phase2/ppo/ppo_best.pt'
if not os.path.exists(ckpt_2):
    ckpt_2 = 'checkpoints_phase2/ppo/ppo_final.pt'

print(f"Evaluating checkpoint: {ckpt_2}")
print("="*60)
results_2 = evaluate_agent(
    algo='ppo',
    checkpoint_path=ckpt_2,
    levels=[('normal', i) for i in range(1, 6)],
    episodes_per_level=20,
    max_steps=400,
    observation_mode='full',
    device_str='auto',
    render=False,
)

agg = results_2['aggregate']
print(f"\n{'='*60}")
print(f"  Phase 2 AGGREGATE SUCCESS: {agg['success_rate']:.1%}")
print(f"  Per-level breakdown:")
for level_key, metrics in results_2['per_level'].items():
    print(f"    {level_key}: {metrics['success_rate']:.0%} success, avg_reward={metrics['avg_reward']:.1f}")
# Check for catastrophic forgetting: L1-3 should remain strong
l13_success = [m['success_rate'] for k, m in results_2['per_level'].items() if k in ('normal-01', 'normal-02', 'normal-03')]
l13_ok = len(l13_success) > 0 and (sum(l13_success) / len(l13_success)) >= 0.50
phase2_ok = agg['success_rate'] >= 0.25
print(f"  L1-3 retained? {'✅ YES' if l13_ok else '❌ NO — catastrophic forgetting, revisit resume settings'}")
print(f"  Ready for Phase 3? {'✅ YES' if (phase2_ok and l13_ok) else '❌ NO — retrain Phase 2 before proceeding'}")
print(f"{'='*60}")

## 4. Phase 3 — Train on Levels 1 to 7

Final fine-tune step: add L6 and L7 to the mix. Settings:
- **Lower LR (5e-5)**: fine-tuning only — don't disturb Phase 2 weights
- **Lower entropy (0.015 → 0.005)**: policy should be largely committed by now
- **`reset_policy_head_on_resume=False`**: same reasoning as Phase 2
- **ICM disabled**: consistent with Phase 2's lesson
- **L1-7 mixed**: always include all prior levels

In [ ]:
import os
from pathlib import Path
from Bobby_Carrot.rl_models.config import PPOConfig, TrainingConfig, ICMConfig, LevelConfig
from Bobby_Carrot.rl_models.ppo import train_ppo

level_config_3 = LevelConfig(
    train_levels=[('normal', i) for i in range(1, 8)],   # L1-7 MIXED
    test_levels=[('normal', 8), ('normal', 9), ('normal', 10)],
)
train_config_3 = TrainingConfig(
    total_timesteps=400_000,
    device='auto',
    checkpoint_dir=Path('checkpoints_phase3'),
    log_dir=Path('logs_phase3'),
    curriculum=False,
    observation_mode='full',
    max_steps_per_episode=400,                 # matches Phase 2 — prevents wander on unseen L6-7
    reward_scale=1.0,
    reset_policy_head_on_resume=False,         # CRITICAL: keep Phase 2 policy head
)
ppo_config_3 = PPOConfig(
    lr=5e-5,                                   # Low LR — fine-tuning only
    entropy_coeff=0.015,
    entropy_min=0.005,                         # Allow near-deterministic policy late
    clip_ratio=0.1,                            # Even tighter than Phase 2 — final fine-tune
    rollout_length=2048,
    n_epochs=4,
    minibatch_size=128,
    cnn_channels=[32, 64, 64, 64],
)
# ICM disabled — same reasoning as Phase 2.
icm_config_3 = ICMConfig(enabled=False)

resume_2 = 'checkpoints_phase2/ppo/ppo_best.pt'
if not os.path.exists(resume_2):
    resume_2 = 'checkpoints_phase2/ppo/ppo_final.pt'

print(f"Loading agent from Phase 2: {resume_2}")
print("Starting Phase 3 Training (L1-7 mixed, 400k steps, policy head PRESERVED)...")
agent_3 = train_ppo(
    ppo_config_3, train_config_3, level_config_3, icm_config_3,
    resume_path=resume_2,
)

### Phase 3 — Evaluation on Training Levels

In [ ]:
import os
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

ckpt_3 = 'checkpoints_phase3/ppo/ppo_best.pt'
if not os.path.exists(ckpt_3):
    ckpt_3 = 'checkpoints_phase3/ppo/ppo_final.pt'

print(f"Evaluating checkpoint: {ckpt_3}")
print("="*60)
results_3_train = evaluate_agent(
    algo='ppo',
    checkpoint_path=ckpt_3,
    levels=[('normal', i) for i in range(1, 8)],
    episodes_per_level=20,
    max_steps=400,
    observation_mode='full',
    device_str='auto',
    render=False,
)

agg = results_3_train['aggregate']
print(f"\n{'='*60}")
print(f"  Phase 3 TRAIN LEVELS AGGREGATE: {agg['success_rate']:.1%}")
for level_key, metrics in results_3_train['per_level'].items():
    print(f"    {level_key}: {metrics['success_rate']:.0%} success")
print(f"{'='*60}")

## 5. Final Evaluation — Test Levels 8-10 (Unseen)
The ultimate test: can the agent generalize to levels it has **never seen**?

In [ ]:
import os
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

ckpt_3 = 'checkpoints_phase3/ppo/ppo_best.pt'
if not os.path.exists(ckpt_3):
    ckpt_3 = 'checkpoints_phase3/ppo/ppo_final.pt'

print("\n" + "="*60)
print("  FINAL TEST: Unseen Levels 8-10")
print("="*60 + "\n")

results_test = evaluate_agent(
    algo='ppo',
    checkpoint_path=ckpt_3,
    levels=[('normal', 8), ('normal', 9), ('normal', 10)],
    episodes_per_level=20,
    max_steps=400,
    observation_mode='full',
    device_str='auto',
    render=False,
)

agg = results_test['aggregate']
target_met = agg['success_rate'] >= 0.40
print(f"\n{'='*60}")
print(f"  AGGREGATE SUCCESS: {agg['success_rate']:.1%}")
print(f"  TARGET (>40%):     {'MET' if target_met else 'NOT MET'}")
print(f"  Per-level breakdown:")
for level_key, metrics in results_test['per_level'].items():
    print(f"    {level_key}: {metrics['success_rate']:.0%} success, avg_reward={metrics['avg_reward']:.1f}")
print(f"{'='*60}")

## 6. Save Final Weights to Drive

In [ ]:
import shutil
import os

drive_dest = '/content/drive/MyDrive/Bobby_Carrot_RL'
os.makedirs(drive_dest, exist_ok=True)

for phase in [1, 2, 3]:
    for name in ['ppo_best.pt', 'ppo_final.pt']:
        src = f'checkpoints_phase{phase}/ppo/{name}'
        if os.path.exists(src):
            dest_name = f'ppo_phase{phase}_{name.replace("ppo_", "")}'
            shutil.copy(src, os.path.join(drive_dest, dest_name))
            print(f"Copied {src} → {dest_name}")

print(f"\nAll saved to {drive_dest}")

try:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Drive flushed.')
except:
    pass
